# Hive -- colony gates (G4-G6, P1, P3)

Runs the full colony under the v2 serving contract (`HIVE_SERVING=v2`) and saves answers, per-item
summaries and logs. **Grading is local** (`colony_gates.py` with `hive_grader.py`; the answer keys never
reach Kaggle).

**Before you start:** GPU **T4 x2** (one is used), **Internet on**, inputs:
1. the **v2.1 adapters** dataset (folders `v2.1-lora-full-lr0.0002-s0` / `-s1`), unless `ARM = 'base'`;
2. the **probe 4 prompts** dataset (the `main_prompts.jsonl` with the 40 dev prompts, tiers 1-4).

**`RUN`** -- one per session, in this order:
- `'time'`: 2 tier 3-4 items -- **run this first**; its seconds per item size everything else.
- `'g6'`: the 20 dev items of tiers 3-4 (G6 + P1), seed `SEED` (2 seeds in all).
- `'g5'`: the school prompt, 3 runs (G5).
- `'g4'`: the 40 dev items of tiers 1-4 with the root as a lone **executor** (G4), one per arm.
- `'judge'`: probe 4's 4 judge cases plus P3's 12 (`probes/judge_cases_p3.py`) through the v2 judge (P3).
  Minutes; can share a session.

**Push first:** the notebook resets to `origin/main`, so everything it uses must be on GitHub.

**`ARM`**: `'v2.1'` (the adapter) or `'base'` (no adapter, same serving). A session that dies can be
re-run with the same settings: answered items are skipped.


In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/arkapravopal04/Swarm_neural_nets.git'
REPO_DIR = '/kaggle/working/Swarm_neural_nets'

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'{cmd} failed: {r.stdout} {r.stderr}')
    return r

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    run(['git', 'clone', REPO_URL, REPO_DIR])
run(['git', '-C', REPO_DIR, 'fetch', 'origin', 'main'])
run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'])
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('synced to:', run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s']).stdout.strip())

In [ ]:
!pip -q install transformers==5.22.2 peft==0.21.2 accelerate==1.15.0 sentence-transformers faiss-cpu sympy

## Config

In [ ]:
import glob, json

RUN = 'time'        # 'time' | 'g6' | 'g5' | 'g4' | 'judge'
ARM = 'v2.1'        # 'v2.1' | 'base'
SEED = 0            # g6: 0 then 1 (two sessions, or both in one if time allows)
TAG, ADAPTER_SEED = 'v2.1', 1   # the adapter folder: {TAG}-lora-full-lr0.0002-s{ADAPTER_SEED}

os.environ['HIVE_SERVING'] = 'v2'   # before anything reads it: the v2 serving contract
if ARM == 'base':
    ADAPTER_PATH = 'none'
else:
    hits = [h for h in glob.glob(f'/kaggle/input/**/{TAG}-lora-full-lr0.0002-s{ADAPTER_SEED}/best/adapter_config.json',
                                 recursive=True) if '/adapters_pack/' not in h]
    assert len(hits) == 1, f'attach the v2.1 adapters dataset (found {len(hits)})'
    ADAPTER_PATH = os.path.dirname(hits[0])

import bench_runner
if RUN in ('time', 'g6', 'g4'):
    found = glob.glob('/kaggle/input/**/main_prompts.jsonl', recursive=True)
    assert len(found) == 1, f'attach the probe 4 prompts dataset (found {found})'
    prompts = bench_runner.load_prompts(found[0])
    dev = json.load(open('bench_splits.json'))['dev']
    tiers = {'time': ('T3', 'T4'), 'g6': ('T3', 'T4'), 'g4': ('T1', 'T2', 'T3', 'T4')}[RUN]
    ITEMS = [prompts[i] for i in dev if i in prompts and i[:2] in tiers]
    if RUN == 'time':
        ITEMS = [next(r for r in ITEMS if r['id'].startswith('T3')), next(r for r in ITEMS if r['id'].startswith('T4'))]
elif RUN == 'g5':
    school = next(c for c in bench_runner.CANARIES if c['id'] == 'canary-school')
    ITEMS = [dict(school, id=f'canary-school-r{k}') for k in range(3)]
else:
    ITEMS = []
want = {'time': 2, 'g6': 20, 'g5': 3, 'g4': 40, 'judge': 0}[RUN]
assert len(ITEMS) == want, f'{RUN} expects {want} items, found {len(ITEMS)} -- check the prompts dataset'
ROOT_ROLE = 'executor' if RUN == 'g4' else 'decomposer'
OUT = f'/kaggle/working/gates/{RUN}-{ARM}-s{SEED}'
os.makedirs(OUT, exist_ok=True)
print(RUN, ARM, 'seed', SEED, '| adapter', ADAPTER_PATH, '| root', ROOT_ROLE, '|', len(ITEMS), 'items ->', OUT)

## Load the model (as `main.py` does)

In [ ]:
import torch
import peft.tuners.lora.torchao as _lora_torchao
_lora_torchao.is_torchao_available = lambda: False  # Kaggle's torchao 0.10 vs this peft

import main
from sentence_transformers import SentenceTransformer
from colony_state import ColonyState
from task_graph import TaskGraph
from event_queue import Messenger
from problem_phaser import Problem_Phaser
from judge import Judge
from memory_state import MemoryStore
from synthesizer import Synthesizer
from orchestrator import Orchestrator

model, tokeniser = main.load_model(ADAPTER_PATH)
embedder = SentenceTransformer(main.EMBED_MODEL_NAME)
judge_fn, synth_fn = main.build_call_fns(model, tokeniser)
import uuid

def new_colony():
    # A fresh memory per item: no item reuses another's accepted answers or lessons,
    # so results do not depend on item order or on where a session resumed.
    memory_store = MemoryStore(ghost_persist_path=os.path.join(OUT, 'ghosts', uuid.uuid4().hex[:8]), embed_model=embedder)
    return Orchestrator(
        ColonyState(initial_budget=0, goal_embedding=None), TaskGraph(), Messenger(),
        phaser=Problem_Phaser(model, tokeniser, embed_model=embedder),
        judge=Judge(llm_call_fn=judge_fn), memory_store=memory_store,
        synthesizer=Synthesizer(llm_call_fn=synth_fn),
        model=model, tokeniser=tokeniser, embed_model=embedder,
        energy_trace_path=os.path.join(OUT, 'energy_trace.jsonl'), root_role=ROOT_ROLE)

# Fail here, not item by item: run_bench records a colony that cannot be built as an
# answered (empty) item, and resume would then skip it for good.
new_colony()
print('ready')

## Run

In [ ]:
if RUN == 'judge':
    from probes.probe4_serving import JUDGE_CASES, SCHOOL
    from probes.judge_cases_p3 import JUDGE_CASES_P3
    judge = Judge(llm_call_fn=judge_fn)
    with open(os.path.join(OUT, 'judge.jsonl'), 'w', encoding='utf-8') as f:
        # probe 4's 4 school cases, then P3's 12 (each with its own task; no project goal).
        for case in JUDGE_CASES + JUDGE_CASES_P3:
            goal = SCHOOL if case in JUDGE_CASES else None
            r = judge.deep_critique(case['output'], case['task'], project_goal=goal,
                                    built_on=case.get('built_on'),
                                    labelled_estimates=case.get('labelled_estimates'))
            row = dict(case=case['id'], expect=case['expect'], verdict=r.get('verdict'),
                       reasoning=(r.get('reasoning') or '')[:1500])
            f.write(json.dumps(row) + '\n')
            print(row['case'], 'expect', row['expect'], '->', row['verdict'])
else:
    rows = bench_runner.run_bench(new_colony, ITEMS, OUT, seed=24 + SEED)
    for r in rows:
        print(r['id'], r['status'], f"{r['seconds']:.0f}s", 'root children', r['root_children'])

## Download

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/gates', 'zip', '/kaggle/working/gates')
print('download /kaggle/working/gates.zip')